In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import date_format
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
def ler_ultima_particao_tabela_spark(spark, source_table):
  """
  Essa função ler a ultima partição das Tabelas no formato delta baaseado na coluna de data_processamento
  """
  try: 
    # Mais performatica para pegar os metadados
    show_partitions_df = spark.sql(f"SHOW PARTITIONS {source_table}")
    # maior partição da data_processamento
    max_partition = show_partitions_df.agg(f.max("data_processamento")).collect()[0][0]
    print(f"partição maxima {max_partition}")

    # pegar o dataframe com maior partição
    return spark.table(f"{source_table}")\
                      .filter(f.col("data_processamento") == max_partition)
  except Exception as e:
    print(f"Erro ao ler o caminho {source_table}: {e}")
    return None


### 1. Taxa Selic

In [0]:
silver_path_selic = "workspace.case_spark_cvm.silver_dados_selic_diario"

df_silver_selic = ler_ultima_particao_tabela_spark(spark, silver_path_selic)

In [0]:
display(df_silver_selic)

### 2. Fundos Diarios

In [0]:
silver_table_cvm_day = "workspace.case_spark_cvm.silver_cvm_fundos_diario"

df_cvm_fundos_diario_silver = ler_ultima_particao_tabela_spark(spark, silver_table_cvm_day)

In [0]:
df_cvm_fundos_diario_silver.columns

In [0]:
gold_fato_diario = df_cvm_fundos_diario_silver\
    .select("cnpj_fundo_classe", "dt_comptc", "vl_quota", "vl_total", "vl_patrim_liq", "captc_dia", "resg_dia", "nr_cotst")

In [0]:
window_spec = Window.partitionBy("cnpj_fundo_classe").orderBy("dt_comptc")

window_spec_first = Window.partitionBy("cnpj_fundo_classe").orderBy("dt_comptc").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

window_21d = Window.partitionBy("cnpj_fundo_classe").orderBy("dt_comptc").rowsBetween(-20, 0)

window_63d = Window.partitionBy("cnpj_fundo_classe").orderBy("dt_comptc").rowsBetween(-62, 0)

window_252d = Window.partitionBy("cnpj_fundo_classe").orderBy("dt_comptc").rowsBetween(-251, 0)

window_inicio = Window.partitionBy("cnpj_fundo_classe").orderBy("dt_comptc").rowsBetween(Window.unboundedPreceding, 0)

window_data_inicio = Window.partitionBy("cnpj_fundo_classe")

gold_fato_diario = gold_fato_diario\
    .withColumn(
        "retorno_diario",
        f.try_divide(f.col("vl_quota"), f.lag("vl_quota", 1).over(window_spec) ) - 1 # retorno do dia em %
    )\
    .withColumn(
        "retorno_21d",
        f.try_divide(f.col("vl_quota"), f.lag("vl_quota", 21).over(window_spec) ) - 1 # retorno ~1 mês
    )\
    .withColumn(
        "retorno_63d",
        f.try_divide(f.col("vl_quota"), f.lag("vl_quota", 63).over(window_spec) ) - 1 # retorno ~3 meses
    )\
    .withColumn(
        "retorno_126d",
        f.try_divide(f.col("vl_quota"), f.lag("vl_quota", 126).over(window_spec) ) - 1 # retorno ~6 meses
    )\
    .withColumn(
        "retorno_252d",
        f.try_divide(f.col("vl_quota"), f.lag("vl_quota", 252).over(window_spec) ) - 1 # retorno ~1 ano
    )\
    .withColumn(
        "retorno_inicio",
        f.try_divide(f.col("vl_quota"), f.first("vl_quota").over(window_spec_first) ) - 1 # retorno desde o início
    )\
    .withColumn(
        "captacao_liquida_dia",
         f.col("captc_dia") - f.col("resg_dia") # captação líquida diária
    )\
    .withColumn(
        "captacao_liquida_21d",
        f.sum("captacao_liquida_dia").over(window_21d) # captação líquida 1 mês
    )\
    .withColumn(
        "captacao_liquida_252d",
        f.sum("captacao_liquida_dia").over(window_252d) # captação líquida 1 ano
    )\
    .withColumn(
        "variacao_cotistas",
        f.try_divide(f.col("nr_cotst"), f.lag("nr_cotst", 1).over(window_spec) ) - 1 # variação diária de cotistas
    )\
    .withColumn(
        "volatilidade_21d",
        f.stddev("retorno_diario").over(window_21d) * f.sqrt(f.lit(252)) # vol anualizada 1 mês
    )\
    .withColumn(
        "volatilidade_63d",
        f.stddev("retorno_diario").over(window_63d) * f.sqrt(f.lit(252)) # vol anualizada 3 meses
    )\
    .withColumn(
        "volatilidade_252d",
        f.stddev("retorno_diario").over(window_252d) * f.sqrt(f.lit(252)) # vol anualizada 1 ano
    )\
    .withColumn(
        "max_quota_historico",
        f.max("vl_quota").over(window_inicio) # pico histórico da cota
    )\
    .withColumn(
        "drawdown",
        f.try_divide(f.col("vl_quota"), f.col("max_quota_historico") ) -1 # queda em relação ao pico
    )\
    .withColumn(
        "drawdown_maximo_252d",
        f.min("drawdown").over(window_252d) # pior drawdown em 1 ano
    )\
    .join(
        df_silver_selic,
        gold_fato_diario.dt_comptc == df_silver_selic.data
    )\
    .drop("data", "data_processamento")\
    .withColumnRenamed("valor", "valor_selic")\
    .withColumn(
        "selic_acum_252d",
        f.sum(f.col("valor_selic") / 100).over(window_252d) # Selic acumulada 1 ano
    )\
    .withColumn(
        "sharpe_252d",
        f.try_divide(f.col("retorno_252d") - f.col("selic_acum_252d"), f.col("volatilidade_252d")) # Sharpe 1 ano
    )\
    .withColumn(
        "retorno_negativo_diario",
        f.when(f.col("retorno_diario") < 0, f.col("retorno_diario")).otherwise(0) # só retornos negativos
    )\
    .withColumn(
        "downside_deviation_252d",
        f.stddev("retorno_negativo_diario").over(window_252d) * f.sqrt(f.lit(252)) # desvio negativo anualizado
    )\
    .withColumn(
        "sortino_252d",
        f.try_divide(f.col("retorno_252d") - f.col("selic_acum_252d"), f.col("downside_deviation_252d")) # Sortino 1 ano
    )\
    .withColumn(
        "var_95_252d",
        f.percentile_approx("retorno_diario", 0.05).over(window_252d) # Sortino 1 ano
    )\
    .withColumn(
        "data_inicio",
        f.min("dt_comptc").over(window_data_inicio) # VaR 95% histórico
    )\
    .withColumn(
        "flag_fundo_novo",
         f.when(f.datediff(f.col("dt_comptc"), f.col("data_inicio")) < 90, "S").otherwise("N") # S/N — fundo com menos de 90 dias
    )\
    .withColumn(
        "flag_resgate_consistente",
        f.when(f.col("captacao_liquida_21d") < 0, "S").otherwise("N") # S/N — saída de dinheiro no mês
    )\
    .drop("data_inicio")

In [0]:
gold_fato_diario = gold_fato_diario.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)



data_proc = int(datetime.now().strftime(f"%Y%m%d"))

gold_fato_diario.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.gold_fato_diario")

In [0]:
display(gold_fato_diario)